# Marketing Campaign Response Modelling

## 01 - Data Profiling & Quality Audit

**Dataset:** UCI Bank Marketing Dataset (Moro et al., 2014)

**Objective:** Predict which customers are most likely to subscribe to a term deposit after a
direct marketing call, so campaign targeting can be improved and marketing cost reduced.


## 0. Environment & Reproducibility

In [1]:
import sys
import hashlib
import json
from pathlib import Path

import pandas as pd
import numpy as np
import scipy

print(f"Python : {sys.version.split()[0]}")
print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")
print(f"scipy  : {scipy.__version__}")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

Python : 3.13.13
pandas : 2.3.3
numpy  : 2.4.4
scipy  : 1.18.0


In [2]:
# RAW_PATH = Path("../data/raw/bank-additional-full.csv")

# if not RAW_PATH.exists():
#     raise FileNotFoundError(
#         f"Raw data file not found at {RAW_PATH.resolve()}.\n"
#         "Download bank-additional-full.csv from the UCI Bank Marketing dataset page and place it "
#         "in data/raw/ before running this notebook."
#     )

# file_hash = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
# print(f"File     : {RAW_PATH}")
# print(f"Size     : {RAW_PATH.stat().st_size:,} bytes")
# print(f"SHA-256  : {file_hash}")

# df = pd.read_csv(RAW_PATH, sep=";")
# df.head()

In [3]:
RAW_PATH = Path(r"D:\Important Files\Wapexp Institute\Project 1\Machine Learning Projects\Marketing Campaign Response Modelling\data\raw\bank-additional-full.csv")

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Raw data file not found at {RAW_PATH.resolve()}.\n"
        "Download bank-additional-full.csv from the UCI Bank Marketing dataset page and place it "
        "in data/raw/ before running this notebook."
    )

file_hash = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
print(f"File     : {RAW_PATH}")
print(f"Size     : {RAW_PATH.stat().st_size:,} bytes")
print(f"SHA-256  : {file_hash}")

df = pd.read_csv(RAW_PATH, sep=";")
df.head()

File     : D:\Important Files\Wapexp Institute\Project 1\Machine Learning Projects\Marketing Campaign Response Modelling\data\raw\bank-additional-full.csv
Size     : 5,834,924 bytes
SHA-256  : 74adfc578bf77a7ff4bb1ba4a9f8709d9e3c6907342959c2c8416847e0afb4d8


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## Business Problem Statement

Marketing campaigns are expensive, and contacting every customer is inefficient only about 1 in
9 calls in this dataset results in a subscription.

The objective is a model that ranks customers by likelihood of subscribing to a term deposit, so
the bank can:

- Improve campaign targeting
- Reduce marketing cost
- Increase conversion rate
- Maximize return on investment
- Support data-driven call-list decisions

This is a **binary classification problem used as a ranking/scoring problem** the deliverable is
a probability score per customer, not just a yes/no label.

### Success criteria (stated now, used later)

Without this, later steps have nothing to anchor to in particular, threshold selection
(`18_threshold_optimization`) is a business-economics decision, not a statistic like F1, and needs
these numbers defined up front, not discovered halfway through the project.

- **Primary model metric: PR-AUC**, with **lift/capture at top-10% and top-20%** of the ranked
  list as the operational metric (imbalanced target, ranking use case accuracy is not the
  target metric).
- **Deployment framing:** the model outputs a ranked call list; the bank calls the top X% of that
  list based on available agent capacity for the period. X is not fixed yet profit simulation in
  a later notebook will inform it.
- **Cost / value inputs still needed from the business** to make the profit simulation real:
  cost per call, and expected value of one subscription. Both are placeholders in
  `src/config.py::COST_PER_CALL` / `PROFIT_PER_SUBSCRIBER` until real bank figures are supplied.

## 1. Dataset Overview

In [4]:
print(f"Rows      : {df.shape[0]:,}")
print(f"Columns   : {df.shape[1]}")
print(f"Features  : {df.shape[1] - 1}")
print(f"Target    : y")

Rows      : 41,188
Columns   : 21
Features  : 20
Target    : y


In [5]:
df.columns.tolist()

['age',
 'job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'duration',
 'campaign',
 'pdays',
 'previous',
 'poutcome',
 'emp.var.rate',
 'cons.price.idx',
 'cons.conf.idx',
 'euribor3m',
 'nr.employed',
 'y']

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

## 2. Numerical vs Categorical Features

Using `select_dtypes(exclude=["number"])` rather than `include="object"`: on pandas versions with
the string-dtype backend (confirmed in this environment - pandas 3.0), text columns can report
dtype `"str"` instead of `"object"`, and `include="object"` then silently misses them. Excluding
numeric types is robust across pandas versions.

In [7]:
target_col = "y"

categorical_cols = df.select_dtypes(exclude=["number"]).columns.tolist()
categorical_cols = [c for c in categorical_cols if c != target_col]
numerical_cols = df.select_dtypes(include=["number"]).columns.tolist()

print("Categorical Features:", len(categorical_cols))
print(categorical_cols)
print("=" * 110)
print("Numerical Features:", len(numerical_cols))
print(numerical_cols)

Categorical Features: 10
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
Numerical Features: 10
['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']


## 3. Missing Values - Standard (NaN)

In [8]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print("No standard NaN values found." if missing.empty else missing)

No standard NaN values found.


## 4. Missing Values - Disguised

Zero `NaN` does not mean zero missing data. This dataset encodes missingness as a **string
category** in several columns. We scan every categorical column for common placeholder tokens.

`"no"` is deliberately excluded from the placeholder list: in `default`, `housing`, and `loan` it
is a real, meaningful answer, not a stand-in for missing data. Flagging every `"no"` as null would
wreck three genuine features.

In [9]:
placeholder_tokens = ["", " ", "na", "n/a", "null", "none", "nan", "?", "unknown", "missing", "-", "--"]

disguised_missing = []
for col in categorical_cols:
    normalized = df[col].astype(str).str.strip().str.lower()
    hits = normalized[normalized.isin(placeholder_tokens)]
    for token, count in hits.value_counts().items():
        disguised_missing.append({
            "column": col, "placeholder": token, "count": count,
            "pct_of_column": round(count / len(df) * 100, 2),
        })

disguised_missing_df = pd.DataFrame(disguised_missing).sort_values("count", ascending=False)
disguised_missing_df

,column,placeholder,count,pct_of_column
3,default,unknown,8597,20.87
2,education,unknown,1731,4.20
4,housing,unknown,990,2.40
5,loan,unknown,990,2.40
0,job,unknown,330,0.80
1,marital,unknown,80,0.19


`"unknown"` appears in 6 columns, from under 0.2% (`marital`) up to ~20.9% (`default`) of rows. These are real missing-data signals encoded as text, not NaN.

## 5. Full Category Levels per Column

`describe()` on categorical data only shows the top value and its frequency not enough to make
an encoding decision. Every level of every categorical column, in full, below. Two things worth
flagging while reading through: `education` has an explicit ordinal hierarchy
(`basic.4y` < `basic.6y` < `basic.9y` < `high.school` < ... < `university.degree`) that a plain
one-hot encoding would throw away, and `month` only has **10 of 12 possible values** no January
or February contacts anywhere in this dataset.

In [10]:
for col in categorical_cols + [target_col]:
    print(f"{col} ({df[col].nunique()} levels)")
    print("="*20)
    print(df[col].value_counts())
    print()

job (12 levels)
job
admin.           10422
blue-collar       9254
technician        6743
services          3969
management        2924
retired           1720
entrepreneur      1456
self-employed     1421
housemaid         1060
unemployed        1014
student            875
unknown            330
Name: count, dtype: int64

marital (4 levels)
marital
married     24928
single      11568
divorced     4612
unknown        80
Name: count, dtype: int64

education (8 levels)
education
university.degree      12168
high.school             9515
basic.9y                6045
professional.course     5243
basic.4y                4176
basic.6y                2292
unknown                 1731
illiterate                18
Name: count, dtype: int64

default (3 levels)
default
no         32588
unknown     8597
yes            3
Name: count, dtype: int64

housing (3 levels)
housing
yes        21576
no         18622
unknown      990
Name: count, dtype: int64

loan (3 levels)
loan
no         33950
yes         6

## 6. Numeric Sentinel Value - `pdays`

`pdays` (days since the customer was last contacted in a *previous* campaign) uses **999** to mean
"never previously contacted." This is not a real day count and is a disguised-missing pattern in a
numeric column invisible to `isnull()`.

In [11]:
sentinel_count = (df["pdays"] == 999).sum()
print(f"pdays == 999: {sentinel_count:,} rows ({sentinel_count/len(df):.1%} of the dataset)")
print()
print("pdays distribution EXCLUDING the sentinel:")
print(df.loc[df["pdays"] != 999, "pdays"].describe())

pdays == 999: 39,673 rows (96.3% of the dataset)

pdays distribution EXCLUDING the sentinel:
count    1515.000000
mean        6.014521
std         3.824906
min         0.000000
25%         3.000000
50%         6.000000
75%         7.000000
max        27.000000
Name: pdays, dtype: float64


96.3% of rows are the sentinel. Must be replaced with a `pdays_missing` flag + NaN/median before any scaling or distance-based modelling - never used as a raw number.

## 7. Cross-Column Consistency - `pdays` ↔ `previous` ↔ `poutcome`

Checking columns one at a time isn't enough these three columns all describe the *same* thing
(prior contact history) and should agree with each other. `pdays == 999` and `poutcome ==
"nonexistent"` should, logically, mark the exact same set of "never contacted before" customers.
Checked directly below rather than assumed.

In [12]:
never_contacted_by_pdays = (df["pdays"] == 999)
never_contacted_by_poutcome = (df["poutcome"] == "nonexistent")

print(f"pdays == 999                : {never_contacted_by_pdays.sum():,} rows")
print(f"poutcome == 'nonexistent'    : {never_contacted_by_poutcome.sum():,} rows")
print()

mismatch = df[never_contacted_by_pdays & ~never_contacted_by_poutcome]
print(f"Rows where pdays=999 (says 'never contacted') "
      f"but poutcome is NOT 'nonexistent' (says there WAS a previous campaign): {len(mismatch):,}")
print(mismatch["poutcome"].value_counts())

reverse_mismatch = df[~never_contacted_by_pdays & never_contacted_by_poutcome]
print(f"\nReverse direction (pdays says contacted, poutcome says never): {len(reverse_mismatch):,}")

pdays == 999                : 39,673 rows
poutcome == 'nonexistent'    : 35,563 rows

Rows where pdays=999 (says 'never contacted') but poutcome is NOT 'nonexistent' (says there WAS a previous campaign): 4,110
poutcome
failure    4110
Name: count, dtype: int64

Reverse direction (pdays says contacted, poutcome says never): 0


In [13]:
# previous vs poutcome — do wo columns jo dono "prior contact" record karte hain
never_contacted_by_previous = (df["previous"] == 0)
print(f"previous == 0                : {never_contacted_by_previous.sum():,} rows")

agreement = (never_contacted_by_previous == never_contacted_by_poutcome).sum()
print(f"previous & poutcome agree on : {agreement:,} / {len(df):,} rows")

# Outcome breakdown of the pdays/poutcome mismatch — is it time-related or outcome-specific?
print("\nOutcome breakdown of the 4,110 mismatched rows:")
print(mismatch["poutcome"].value_counts())

previous == 0                : 35,563 rows
previous & poutcome agree on : 41,188 / 41,188 rows

Outcome breakdown of the 4,110 mismatched rows:
poutcome
failure    4110
Name: count, dtype: int64


**Real, genuine data inconsistency: 4,110 rows (10% of the data)** have `pdays == 999` but
`poutcome == "failure"` the sentinel says "never contacted before," but a previous-campaign
outcome is recorded regardless. The reverse never happens (0 rows), which is itself informative.

**A plausible explanation** (stated as a hypothesis, not a confirmed fact the UCI documentation
doesn't clarify this): `pdays` may only measure contacts within a bounded recent window, while
`poutcome` records the result of *any* previous campaign regardless of how long ago. A customer
contacted much earlier than `pdays` can represent would show `pdays=999` (out of the measurable
window) while still carrying a recorded `poutcome`. This can't be confirmed from the data alone
flagged here as a known inconsistency to carry into feature engineering (e.g. `previous_contact`
should probably be derived from `previous > 0` or `poutcome != "nonexistent"`, not from `pdays`
alone, exactly because `pdays` is shown here to be the less reliable of the two signals).

**Cross-check confirms it:** `previous` and `poutcome` agree on **41,188 / 41,188 rows (100%)** these two columns are fully consistent with each other. `pdays` is the only column that disagrees with the other two, which is direct evidence (not just inference) that `pdays` is the less reliable signal of prior contact.

The mismatch is also outcome-specific, not random: all 4,110 mismatched rows have `poutcome == "failure"` zero have `poutcome == "success"`. If the cause were purely a bounded time-window on `pdays`, old *successful* contacts should show the same pattern. They don't, which narrows (without fully confirming) the earlier time-window hypothesis.

## 8. Duplicates - Checked Two Ways

A single `.duplicated().sum()` is not enough on this dataset.

In [14]:
exact_dupes = df.duplicated().sum()
print(f"Exact full-row duplicates (all 21 columns): {exact_dupes}")

dupes_no_duration = df.drop(columns=["duration"]).duplicated().sum()
print(f"Duplicates ignoring 'duration':              {dupes_no_duration}  ({dupes_no_duration/len(df):.1%})")

Exact full-row duplicates (all 21 columns): 12
Duplicates ignoring 'duration':              1784  (4.3%)


In [15]:
dupe_mask = df.drop(columns=["duration"]).duplicated(keep=False)
example_key = df.loc[dupe_mask].drop(columns=["duration"]).iloc[0]
example_matches = df.drop(columns=["duration"])[(df.drop(columns=["duration"]) == example_key).all(axis=1)]
print(f"{len(example_matches)} rows identical on every column except duration:")
df.loc[example_matches.index]

2 rows identical on every column except duration:


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
5592,37,services,married,high.school,no,yes,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [16]:
dupe_mask_incl = df.drop(columns=["duration"]).duplicated(keep=False)

print("Target rate WITHIN the 1,784-row duplicate group:")
print(df.loc[dupe_mask_incl, "y"].value_counts(normalize=True).round(3))
print()
print("Target rate in the REST of the data:")
print(df.loc[~dupe_mask_incl, "y"].value_counts(normalize=True).round(3))

Target rate WITHIN the 1,784-row duplicate group:
y
no     0.975
yes    0.025
Name: proportion, dtype: float64

Target rate in the REST of the data:
y
no     0.88
yes    0.12
Name: proportion, dtype: float64


**Two numbers, and neither alone is "the" answer both interpretations need to be on the table:**

- All 21 columns match: **12** duplicates. This undercounts, because `duration` (call length in
  seconds) is close to unique per call and prevents two genuinely repeated records from matching.
- Excluding `duration`: **1,784 rows (4.3%)** match exactly.

**But 1,784 is not automatically "the honest number" either it needs a caveat.** This dataset has
**no customer ID**. With only 20 mostly low-cardinality columns (`job` has 12 levels, `education`
has 8, etc.), it is statistically plausible that two genuinely *different* customers — say, two
different 37-year-old married people in services jobs, both contacted the same week under the same
macro conditions with the same campaign history end up identical on every visible column by
coincidence, not because a call was logged twice.

**Both readings are legitimate:**
1. *"These are logging duplicates"* → drop them, 1,784 rows.
2. *"These are different customers who happen to share every recorded attribute"* → keep them,
   because dropping would be discarding real, distinct observations and shrinking the (already
   imbalanced) positive class.

This notebook documents both counts and the reasoning; the actual drop/keep decision belongs in
`06_data_cleaning_preprocessing`, ideally revisited after checking whether the dropped rows skew
the target rate noticeably (a hint one way or the other).

**New evidence that leans the decision one way:** the positive rate *inside* the 1,784-row duplicate group is **2.5%**, versus **12%** in the rest of the data nearly a 5x gap. If these rows were genuinely different customers who coincidentally matched on every column, their target rate should look like the rest of the dataset, not diverge this sharply. This tilts the evidence toward "these are more likely logging duplicates" rather than coincidental matches, though the final drop/keep decision is still deferred to `06_data_cleaning_preprocessing`.

## 9. Cardinality & Near-Constant / Rare-Category Features

In [17]:
cardinality = df.nunique().sort_values(ascending=False)
pd.DataFrame({"n_unique": cardinality, "pct_unique": (cardinality/len(df)*100).round(3)})

,n_unique,pct_unique
duration,1544,3.749
euribor3m,316,0.767
age,78,0.189
campaign,42,0.102
pdays,27,0.066
cons.price.idx,26,0.063
cons.conf.idx,26,0.063
job,12,0.029
nr.employed,11,0.027
emp.var.rate,10,0.024


In [18]:
print("Columns with a category under 0.5% of the data:\n")
for col in categorical_cols:
    counts = df[col].value_counts(normalize=True)
    rare = counts[counts < 0.005]
    if len(rare) > 0:
        print("="*40)
        print(f"{col}:")
        print(rare)
        print()

Columns with a category under 0.5% of the data:

marital:
marital
unknown    0.001942
Name: proportion, dtype: float64

education:
education
illiterate    0.000437
Name: proportion, dtype: float64

default:
default
yes    0.000073
Name: proportion, dtype: float64

month:
month
dec    0.004419
Name: proportion, dtype: float64



No ID-like high-cardinality columns. **`default` is a near-constant / rare-category feature**:
`no` (79.1%) and `unknown` (20.9%) dominate, while `yes` occurs in only **3 rows out of 41,188
(0.007%)**. Any model finding strong signal in `default=yes` specifically is fitting 3 examples.

## 10. Numeric Ranges & Extreme Values

Every numeric column's min/max, with the extreme values shown and commented on individually rather
than asserted as "no issues found" without evidence.

In [19]:
ranges = pd.DataFrame({"min": df[numerical_cols].min(), "max": df[numerical_cols].max()})
ranges

,min,max
age,17.000,98.000
duration,0.000,4918.000
campaign,1.000,56.000
pdays,0.000,999.000
previous,0.000,7.000
emp.var.rate,-3.400,1.400
cons.price.idx,92.201,94.767
cons.conf.idx,-50.800,-26.900
euribor3m,0.634,5.045
nr.employed,4963.600,5228.100


In [20]:
print("Rows with the most extreme values on key columns:\n")

print("Highest `campaign` (contacts in this campaign):")
print(df.nlargest(3, "campaign")[["age", "job", "campaign", "poutcome", "y"]])
print()

print("Highest `duration` (call length):")
top_duration = df.nlargest(3, "duration")[["age", "job", "duration", "y"]].copy()
top_duration["duration_minutes"] = (top_duration["duration"] / 60).round(1)
print(top_duration)
print()

print("Age range:")
print(df.nlargest(2, "age")[["age", "job", "y"]])
print(df.nsmallest(2, "age")[["age", "job", "y"]])

Rows with the most extreme values on key columns:

Highest `campaign` (contacts in this campaign):
       age         job  campaign     poutcome   y
4107    32      admin.        56  nonexistent  no
13447   32  technician        43  nonexistent  no
18728   54      admin.        43  nonexistent  no

Highest `duration` (call length):
       age          job  duration    y  duration_minutes
24091   33   technician      4918   no              82.0
22192   52  blue-collar      4199  yes              70.0
40537   27       admin.      3785   no              63.1

Age range:
       age      job    y
38452   98  retired  yes
38455   98  retired  yes
       age      job   y
37140   17  student  no
37539   17  student  no


- `campaign` max = **56** contacts to a single customer in one campaign. Plausible but extreme -
  worth capping or flagging as an engineered feature later (e.g. `campaign_capped`), not dropping;
  the row itself isn't invalid, just an outlier worth being aware of before it dominates a
  distance-based model or a naive linear coefficient.
- `duration` max = **4,918 seconds (~82 minutes)** for a single phone call. Long but not
  impossible for a detailed sales conversation. Irrelevant for modelling regardless, since
  `duration` is excluded entirely as leakage (Section 11), but worth knowing it exists if `duration`
  is ever used for a *different* purpose (e.g. call-center operations analysis).
- `age` range 17-98 plausible for a bank's customer base, no negative or triple-digit
  impossible values. No action needed here, stated with evidence rather than assumed.

## 11. Target Leakage - `duration`

The most important finding in this dataset. `duration` is the length of the phone call in
seconds — only known **after** the call ends, by which point the outcome has already happened.
That makes it leakage by definition, proven numerically below rather than just asserted.

In [21]:
zero_duration = df[df["duration"] == 0]
print(f"Rows with duration == 0 (call never connected): {len(zero_duration)}")
print(zero_duration["y"].value_counts())
print()
print("Mean duration by outcome:")
print(df.groupby("y")["duration"].mean().round(1))
print()
y_numeric = (df["y"] == "yes").astype(int)
print(f"Correlation(duration, y) = {df['duration'].corr(y_numeric):.3f}")

Rows with duration == 0 (call never connected): 4
y
no    4
Name: count, dtype: int64

Mean duration by outcome:
y
no     220.8
yes    553.2
Name: duration, dtype: float64

Correlation(duration, y) = 0.405


Every call with `duration == 0` resulted in `y == "no"`, successful calls average ~550 seconds
vs. ~221 for unsuccessful ones, and the correlation with the target (0.41) is far higher than any
legitimate pre-call feature. **`duration` must be dropped before any modelling.**

## 12. Time Structure a Profiling Finding, Not Just an EDA Topic

This belongs here, not only in the EDA notebook: it changes how columns should be classified, which
is a profiling-stage decision, not just a visualization one. Look at the cardinality of the
macro-economic columns specifically.

In [22]:
macro_cols = ["emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]
for col in macro_cols:
    print(f"{col:16s} {df[col].nunique():4d} unique values")

emp.var.rate       10 unique values
cons.price.idx     26 unique values
cons.conf.idx      26 unique values
euribor3m         316 unique values
nr.employed        11 unique values


In [23]:
# Prove the frequency claim rather than inferring it from nunique() alone
print("Unique values of each macro column, WITHIN each month:\n")
print(df.groupby("month")[macro_cols].nunique())

Unique values of each macro column, WITHIN each month:

       emp.var.rate  cons.price.idx  cons.conf.idx  euribor3m  nr.employed
month                                                                     
apr               1               2              2         29            2
aug               3               3              3         39            3
dec               2               2              2         22            2
jul               3               3              3         51            3
jun               3               3              3         44            3
mar               1               2              2         37            2
may               2               3              3         35            3
nov               3               3              3         38            3
oct               3               3              3         48            3
sep               2               2              2         31            2


`emp.var.rate` (10 values) and `nr.employed` (11 values) update at roughly **quarterly**
frequency; `cons.price.idx` / `cons.conf.idx` (26 values each) at roughly **monthly**; `euribor3m`
(316 values) at roughly **daily**. These are macro-economic time-series values, repeated across
every customer contacted in the same period **they are not independent customer attributes**,
they are a disguised timestamp at different resolutions.

Combined with `month` only covering 10 of 12 months (Section 5) and no explicit date/year column
existing anywhere in the raw data, this dataset spans a **known real-world period (May 2008 -
November 2010, per the UCI documentation) without giving that structure to us directly** - it has
to be reconstructed from these proxies.

**Why this belongs in profiling, not just EDA:** it changes how these 5 columns should be
*classified* for modelling purposes (customer signal vs. time signal), which affects feature
selection and train/test split strategy before a single distribution plot is drawn. The actual
visualization of the trend (subscription rate drifting as these values change) is deferred to EDA
- that's a chart, this is a structural fact about the columns.

## 12.1 Is the Data Actually Row-Ordered by Time?

Section 12 established that time structure *exists* in this dataset. That alone doesn't tell us
whether the rows are stored in chronological order and that answer decides whether a random
train/test split is safe or whether it leaks future information into training (a decision that
belongs in `08_train_validation_test_split`, but the evidence for it belongs here).

In [24]:
print("First 10 months in the file:", df["month"].head(10).tolist())
print("Last 10 months in the file :", df["month"].tail(10).tolist())
print()
print("nr.employed monotonic (non-strict) check across the file:")
print("  Strictly non-increasing:", df["nr.employed"].is_monotonic_decreasing)
print("  Strictly non-decreasing:", df["nr.employed"].is_monotonic_increasing)
print()

# Coarse check: does euribor3m drift as we move through row index?
chunk_means = df.groupby(df.index // 4000)["euribor3m"].mean()
print("euribor3m mean per ~4,000-row block (row-index order):")
print(chunk_means)

First 10 months in the file: ['may', 'may', 'may', 'may', 'may', 'may', 'may', 'may', 'may', 'may']
Last 10 months in the file : ['nov', 'nov', 'nov', 'nov', 'nov', 'nov', 'nov', 'nov', 'nov', 'nov']

nr.employed monotonic (non-strict) check across the file:
  Strictly non-increasing: False
  Strictly non-decreasing: False

euribor3m mean per ~4,000-row block (row-index order):
0     4.857139
1     4.858622
2     4.934984
3     4.960123
4     4.964409
5     4.964032
6     3.933832
7     1.381838
8     1.277910
9     0.884559
10    0.912400
Name: euribor3m, dtype: float64


The evidence says the rows **are** chronologically ordered, not shuffled: `month` moves cleanly from `may` at the start of the file to `nov` at the end, and the `euribor3m` block-means hold steady around **4.86-4.96** for the first six blocks before dropping sharply to **3.93 -> 1.38 -> 1.28 -> 0.88 -> 0.91** a pattern that lines up with the real 2008 financial-crisis interest-rate collapse, not something a shuffled file would produce.

**Consequence carried forward to `08_train_validation_test_split`:** a random train/test split would let the model train on rows from *after* the test period's macro-economic regime, which is a leakage risk in a slightly different form than `duration`. The split in that notebook should be time-based (e.g. a chronological cutoff), not a random shuffle.

## 13. Target Distribution

In [25]:
target_summary = pd.DataFrame({
    "Count": df["y"].value_counts(),
    "Percentage": (df["y"].value_counts(normalize=True) * 100).round(2),
})
target_summary

,Count,Percentage
y,,
no,36548,88.73
yes,4640,11.27


~11% positive class - imbalanced binary classification, consistent with the PR-AUC / lift-based success criteria already stated in the Business Problem section above.

## 14. Statistical Summary

In [26]:
df[numerical_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
age,41188.0,40.024060,10.421250,17.000,32.000,38.000,47.000,98.000
duration,41188.0,258.285010,259.279249,0.000,102.000,180.000,319.000,4918.000
campaign,41188.0,2.567593,2.770014,1.000,1.000,2.000,3.000,56.000
pdays,41188.0,962.475454,186.910907,0.000,999.000,999.000,999.000,999.000
previous,41188.0,0.172963,0.494901,0.000,0.000,0.000,0.000,7.000
emp.var.rate,41188.0,0.081886,1.570960,-3.400,-1.800,1.100,1.400,1.400
cons.price.idx,41188.0,93.575664,0.578840,92.201,93.075,93.749,93.994,94.767
cons.conf.idx,41188.0,-40.502600,4.628198,-50.800,-42.700,-41.800,-36.400,-26.900
euribor3m,41188.0,3.621291,1.734447,0.634,1.344,4.857,4.961,5.045
nr.employed,41188.0,5167.035911,72.251528,4963.600,5099.100,5191.000,5228.100,5228.100


In [27]:
df[categorical_cols + [target_col]].describe().T

,count,unique,top,freq
job,41188,12,admin.,10422
marital,41188,4,married,24928
education,41188,8,university.degree,12168
default,41188,3,no,32588
housing,41188,3,yes,21576
loan,41188,3,no,33950
contact,41188,2,cellular,26144
month,41188,10,may,13769
day_of_week,41188,5,thu,8623
poutcome,41188,3,nonexistent,35563


## 15. Automated Profiling Report (optional)

`ydata_profiling` as a supplement, not a replacement, for the manual checks above - it would not
have caught the `duration` leakage, the `pdays`/`poutcome` inconsistency, or the macro-column time
structure; those need domain reasoning. Wrapped so the notebook still runs cleanly without the
package installed. Slow (several minutes) - optional to run.

In [28]:
try:
    from ydata_profiling import ProfileReport

    Path("../reports/profiling").mkdir(parents=True, exist_ok=True)

    profile = ProfileReport(df, title="Marketing Campaign Profiling Report", explorative=True)
    profile.to_file("../reports/profiling/marketing_profile.html")
    print("Saved -> ../reports/profiling/marketing_profile.html")
except ImportError:
    print("ydata_profiling not installed - skipping (optional). Install with:")
    print("  pip install ydata-profiling")

ydata_profiling not installed - skipping (optional). Install with:
  pip install ydata-profiling


## 16. Key Findings

- 41,188 rows, 20 features + target, mix of numerical and categorical.
- Binary classification, target imbalanced at ~11% positive. Success metric: PR-AUC + lift@10%/20%
  (stated in Business Problem section).
- **No standard `NaN` values** - but 6 columns carry a disguised-missing `"unknown"` category
  (0.2%-20.9% of rows).
- **`pdays` uses 999 as a sentinel** for "never previously contacted" in 96.3% of rows.
- **`pdays` and `poutcome` disagree on 4,110 rows (10%)** - a genuine cross-column data
  inconsistency, not just a single-column quirk. `previous`/`poutcome` are the more reliable
  signal of prior contact than `pdays` alone.
- **Duplicate count depends on whether `duration` is included**: 12 exact-row duplicates vs.
  1,784 (4.3%) excluding `duration` - and even 1,784 is not certain to be true duplication given
  there's no customer ID (see Section 8 for the full reasoning both ways).
- **`default` is a near-constant / rare-category feature** (`yes` in only 3 of 41,188 rows).
- Extreme-but-plausible numeric values exist (`campaign` up to 56, `duration` up to ~82 minutes) -
  documented with evidence, not dropped.
- **The five macro-economic columns are a disguised timestamp** at three different update
  frequencies (quarterly/monthly/daily) - not independent customer attributes. Combined with
  `month` missing Jan/Feb entirely, this dataset has real time structure with no explicit date
  column.
- **`duration` is proven target leakage** (duration=0 -> always "no"; correlation 0.41) and must
  be dropped before modelling.
- **`previous` and `poutcome` agree on 100% of rows (41,188/41,188)** - confirming `pdays` is the only unreliable prior-contact signal of the three, not just the odd one out by assumption. The `pdays`/`poutcome` mismatch is also outcome-specific (100% `failure`, 0% `success`), not random.
- **The duplicate rows (excl. `duration`) show a target rate of 2.5% vs. 12% for the rest of the data** - evidence leaning toward these being logging duplicates rather than coincidental distinct customers, though the drop/keep decision is still deferred to notebook 06.
- **Rows are chronologically ordered, not shuffled** - confirmed via `month` progression (may -> nov) and a matching real-world drop in `euribor3m` around the 2008 financial crisis. A random train/test split would leak future macro-conditions into training; the split strategy in `08_train_validation_test_split` must be time-based.
- No ID-like high-cardinality columns, no inconsistent category spelling found.

## 17. Machine-Readable Profile Output

The findings above, as a structured `data_profile.json` that later notebooks import instead of
re-deriving or re-hardcoding the same lists (and risking them drifting out of sync).

In [29]:
data_profile = {
    "source_file": str(RAW_PATH),
    "source_sha256": file_hash,
    "n_rows": int(df.shape[0]),
    "n_columns": int(df.shape[1]),
    "target_col": target_col,
    "target_positive_label": "yes",
    "target_positive_rate": float((df["y"] == "yes").mean()),
    "categorical_cols": categorical_cols,
    "numerical_cols": numerical_cols,
    "leakage_cols": ["duration"],
    "sentinels": {"pdays": 999},
    "disguised_missing_token": "unknown",
    "disguised_missing_columns": sorted(disguised_missing_df["column"].unique().tolist()),
    "near_constant_columns": {"default": {"rare_category": "yes", "count": int((df["default"] == "yes").sum())}},
    "known_inconsistencies": [
        {
            "columns": ["pdays", "poutcome"],
            "description": "pdays==999 (never contacted) but poutcome != 'nonexistent' (has a recorded prior outcome)",
            "affected_rows": int(len(mismatch)),
        }
    ],
    "macro_economic_cols": macro_cols,
    "duplicate_rows_all_columns": int(exact_dupes),
    "duplicate_rows_excluding_duration": int(dupes_no_duration),
    "success_metric": "PR-AUC + lift@10%/20%",
}

Path("../reports").mkdir(parents=True, exist_ok=True)
with open("../reports/data_profile.json", "w") as f:
    json.dump(data_profile, f, indent=2)

print("Saved -> ../reports/data_profile.json")
data_profile

Saved -> ../reports/data_profile.json


{'source_file': 'D:\\Important Files\\Wapexp Institute\\Project 1\\Machine Learning Projects\\Marketing Campaign Response Modelling\\data\\raw\\bank-additional-full.csv',
 'source_sha256': '74adfc578bf77a7ff4bb1ba4a9f8709d9e3c6907342959c2c8416847e0afb4d8',
 'n_rows': 41188,
 'n_columns': 21,
 'target_col': 'y',
 'target_positive_label': 'yes',
 'target_positive_rate': 0.11265417111780131,
 'categorical_cols': ['job',
  'marital',
  'education',
  'default',
  'housing',
  'loan',
  'contact',
  'month',
  'day_of_week',
  'poutcome'],
 'numerical_cols': ['age',
  'duration',
  'campaign',
  'pdays',
  'previous',
  'emp.var.rate',
  'cons.price.idx',
  'cons.conf.idx',
  'euribor3m',
  'nr.employed'],
 'leakage_cols': ['duration'],
 'sentinels': {'pdays': 999},
 'disguised_missing_token': 'unknown',
 'disguised_missing_columns': ['default',
  'education',
  'housing',
  'job',
  'loan',
  'marital'],
 'near_constant_columns': {'default': {'rare_category': 'yes', 'count': 3}},
 'known_i

## 18. Save Human-Readable Artifacts

In [30]:
dataset_info = pd.DataFrame({
    "Feature": df.columns,
    "Data Type": df.dtypes.astype(str).values,
})
dataset_info.to_csv("../reports/dataset_information.csv", index=False)

target_summary.to_csv("../reports/target_summary.csv")

quality_summary = pd.DataFrame([
    {"issue": "Standard NaN",                   "finding": "None"},
    {"issue": "Disguised missing ('unknown')",   "finding": "6 columns, 0.2%-20.9%"},
    {"issue": "pdays sentinel (999)",            "finding": "96.3% of rows"},
    {"issue": "pdays <-> poutcome inconsistency","finding": f"{len(mismatch)} rows (10%) disagree"},
    {"issue": "Exact duplicates (all cols)",     "finding": f"{exact_dupes}"},
    {"issue": "Duplicates (excl. duration)",     "finding": f"{dupes_no_duration} (4.3%) - both readings documented, decision deferred"},
    {"issue": "Near-constant feature",           "finding": "default: 'yes' in 3/41,188 rows"},
    {"issue": "Macro columns are a time proxy",  "finding": "3 update frequencies, no explicit date column"},
    {"issue": "Target leakage",                  "finding": "duration - must drop before modelling"},
])
quality_summary.to_csv("../reports/data_quality_summary.csv", index=False)

print("Saved: dataset_information.csv, target_summary.csv, data_quality_summary.csv, data_profile.json -> ../reports/")

Saved: dataset_information.csv, target_summary.csv, data_quality_summary.csv, data_profile.json -> ../reports/


## Conclusion

The dataset has been loaded and profiled, including validating relationships *between* columns,
not just checking each one in isolation. Every finding is backed by a number computed directly
from the data, and two findings the `pdays`/`poutcome` inconsistency and the macro-columns' time
structure are only visible once columns are checked against each other rather than one at a
time.

**Open item carried forward:** whether the 1,784 excl.-duration duplicates are logging duplicates
or coincidentally-identical distinct customers is not resolved here - both readings are documented
with reasoning, and the decision belongs in the cleaning notebook, informed by checking whether
dropping them shifts the target rate.

**Next notebook: Exploratory Data Analysis** - univariate -> bivariate -> multivariate, including
class-imbalance visualization, feature distributions, correlation/multicollinearity analysis, and
a direct visualization of the time trend whose *structural* existence was established here in
Section 12 (this notebook found the macro columns are a disguised timestamp; the next one plots
what actually happens - subscription rate - across that timeline).